# Advanced Experiment 1: Baseline and Feature Engineering
Tujuan Eksperimen:
- Menyelaraskan data lingkungan dari resolusi 1 jam menjadi 3 jam.
- Melakukan Feature Engineering tingkat lanjut (Cyclical Time, Rolling Statistics).
- Menerapkan strategi validasi Time Series Split untuk menghindari kebocoran data.
- Melatih model LightGBM dan menyimpan prediksi akhir dalam format submission standar.

In [1]:
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 150)

## 1. Load Data
Membaca dataset utama dan data pendukung dari direktori raw.

In [2]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
env_data = pd.read_csv('../data/raw/data_pendukung/data_lingkungan.csv')
coords = pd.read_csv('../data/raw/data_pendukung/koordinat_pos.csv')

train['datetime'] = pd.to_datetime(train['datetime'])
env_data['datetime'] = pd.to_datetime(env_data['datetime'])

test['datetime'] = pd.to_datetime(test['id'].str[:19])
test['nama_pos'] = test['id'].str[22:]

## 2. Preprocessing & Agregasi
Mengonversi resolusi data eksogen dari 1 jam menjadi 3 jam menggunakan nilai representatif (rata-rata atau agregat temporal).

In [3]:
def aggregate_env_data(df):
    df_sorted = df.sort_values(by=['nama_pos', 'datetime'])
    
    cat_cols = ['nama_pos', 'landcover_name', 'datetime']
    num_cols = [c for c in df.columns if c not in cat_cols]
    
    agg_funcs = {col: 'mean' for col in num_cols}
    agg_funcs['rainfall_mm'] = 'sum'
    agg_funcs['rainfall_openmeteo_mm'] = 'sum'
    agg_funcs['rainfall_max_24h_mm'] = 'max'
    
    df_indexed = df_sorted.set_index('datetime')
    agg_df = df_indexed.groupby(['nama_pos', pd.Grouper(freq='3h', label='right', closed='right')]).agg(agg_funcs).reset_index()
    return agg_df

env_agg = aggregate_env_data(env_data)

env_agg = env_agg.sort_values(['nama_pos', 'datetime'])
env_agg = env_agg.groupby('nama_pos', group_keys=False).apply(lambda group: group.ffill().bfill())

## 3. Merging Dataset
Menggabungkan data target, data lingkungan, dan koordinat pos.

In [4]:
train_df = pd.merge(train, env_agg, on=['datetime', 'nama_pos'], how='left')
test_df = pd.merge(test, env_agg, on=['datetime', 'nama_pos'], how='left')

train_df = pd.merge(train_df, coords, on='nama_pos', how='left')
test_df = pd.merge(test_df, coords, on='nama_pos', how='left')

## 4. Feature Engineering
Membuat fitur baru berdasarkan siklus waktu (cyclical features) dan transformasi lainnya.

In [5]:
def engineer_features(df):
    df['month'] = df['datetime'].dt.month
    df['day'] = df['datetime'].dt.day
    df['hour'] = df['datetime'].dt.hour
    df['dayofweek'] = df['datetime'].dt.dayofweek
    
    df['sin_hour'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['cos_hour'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
    df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)
    
    df = df.sort_values(by=['nama_pos', 'datetime'])
    df['rainfall_rolling_12h_sum'] = df.groupby('nama_pos')['rainfall_mm'].transform(lambda x: x.rolling(window=4, min_periods=1).sum())
    df['rainfall_rolling_24h_sum'] = df.groupby('nama_pos')['rainfall_mm'].transform(lambda x: x.rolling(window=8, min_periods=1).sum())
    
    return df

train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

le = LabelEncoder()
train_df['nama_pos_encoded'] = le.fit_transform(train_df['nama_pos'])
test_df['nama_pos_encoded'] = le.transform(test_df['nama_pos'])

drop_cols = ['datetime', 'nama_pos', 'tma_mdpl', 'id', 'landcover_name']
features = [c for c in train_df.columns if c not in drop_cols]
target = 'tma_mdpl'

## 5. Modeling & Validation
Penerapan skema Time Series Split (70/30) dan melatih global model menggunakan LightGBM.

In [6]:
train_df = train_df.sort_values('datetime')
split_idx = int(len(train_df) * 0.7)

X_train, y_train = train_df.iloc[:split_idx][features], train_df.iloc[:split_idx][target]
X_val, y_val = train_df.iloc[split_idx:][features], train_df.iloc[split_idx:][target]

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.03,
    'num_leaves': 63,
    'max_depth': 8,
    'feature_fraction': 0.8,
    'random_state': 42,
    'n_estimators': 1500,
    'verbose': -1
}

model = lgb.LGBMRegressor(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=100)]
)

val_preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, val_preds))
print("Local Validation RMSE:", rmse)

Training until validation scores don't improve for 100 rounds


Early stopping, best iteration is:
[1208]	valid_0's rmse: 4.54665


Local Validation RMSE: 4.546650643169082


## 6. Full Training & Submission
Melatih ulang model menggunakan seluruh data train untuk diprediksi pada dataset test.

In [7]:
full_model = lgb.LGBMRegressor(**params)
full_model.fit(train_df[features], train_df[target])

test_preds = full_model.predict(test_df[features])

sub = pd.DataFrame({
    'id': test_df['id'],
    'tma_mdpl': test_preds
})

if not os.path.exists('../submissions'):
    os.makedirs('../submissions')

sub.to_csv('../submissions/submission.csv', index=False)
print("File submission.csv berhasil disimpan.")

File submission.csv berhasil disimpan.
